<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [2]:
!pip install -q duckdb

In [3]:
%pip -q install duckdb huggingface_hub


In [4]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("Token found:", bool(token))

Token found: True


In [5]:
from huggingface_hub import login

login(token=token)

In [6]:
from huggingface_hub import whoami

print(whoami(token=token))

{'type': 'user', 'id': '6a5598ed53072296fae6e031', 'name': 'vaishnavikabbe27', 'fullname': 'Vaishnavi Kabbe', 'email': 'vaishnavikabbe@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/3fcfd0e5e3ad95d51ce48389f5d2b062.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank-colab', 'role': 'read', 'createdAt': '2026-08-23T04:47:21.009Z'}}}


In [7]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token: ")

print("HF_TOKEN exists:", bool(os.environ.get("HF_TOKEN")))

Enter your Hugging Face token: ··········
HF_TOKEN exists: True


In [8]:
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset"
)

print(path)

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet


In [10]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset"
)

print("Downloaded successfully:")
print(path)

Downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet


In [11]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN 'hf_YOUR_TOKEN_HERE'
)
""")

In [10]:
import os
from getpass import getpass

HF_TOKEN = getpass("Paste your Hugging Face token: ")
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN exists:", bool(HF_TOKEN))
print("HF_TOKEN starts with hf_:", HF_TOKEN.startswith("hf_"))

Paste your Hugging Face token: ··········
HF_TOKEN exists: True
HF_TOKEN starts with hf_: True


In [15]:
print("HF_TOKEN exists:", bool(HF_TOKEN))
print("HF_TOKEN starts with hf_:", HF_TOKEN.startswith("hf_"))
print("HF_TOKEN length:", len(HF_TOKEN))

HF_TOKEN exists: True
HF_TOKEN starts with hf_: True
HF_TOKEN length: 37


In [16]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN '" + HF_TOKEN + "')"
)

print("DuckDB Hugging Face secret created successfully.")

DuckDB Hugging Face secret created successfully.


In [17]:
result = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
    )
    LIMIT 5
""").df()

print(result)

            client_hash_id  is_active  has_gsc_access  has_ga4_access  \
0  client_04660893ae39614a       True            True            True   
1  client_05475c07ed21a83a       True           False           False   
2  client_06d356715a8ff3b6       True            True            True   
3  client_0797ff3a1fc9a6a5       True           False           False   
4  client_08a6a72ff48e62c0       True            True           False   

                  access_profile client_created_date client_updated_date  \
0                    gsc_and_ga4          2026-04-15          2026-06-27   
1  no_search_or_analytics_access          2026-04-01          2026-06-27   
2                    gsc_and_ga4          2026-03-23          2026-07-05   
3  no_search_or_analytics_access          2025-05-26          2026-06-27   
4                       gsc_only          2025-05-26          2026-06-27   

  gsc_data_start ga4_data_start  
0            NaT     2026-05-22  
1            NaT            NaT  
2 

In [18]:
print(
    con.sql("""
        SELECT COUNT(*)
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
        )
    """).fetchone()[0]
)

104


In [19]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [20]:
con.sql("""
CREATE OR REPLACE VIEW dim_clients AS
SELECT * FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
);

CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
);

CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
);

CREATE OR REPLACE VIEW fact_query_90d AS
SELECT * FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
);
""")

print("Views created successfully.")

Views created successfully.


In [21]:
print("DIM_CLIENTS")
con.sql("DESCRIBE dim_clients").show()

print("\nDIM_CONTENT")
con.sql("DESCRIBE dim_content").show()

print("\nFACT_DAILY")
con.sql("DESCRIBE fact_daily").show()

print("\nFACT_QUERY_90D")
con.sql("DESCRIBE fact_query_90d").show()

DIM_CLIENTS
┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES     │ NULL

In [22]:
print("=== DIM_CLIENTS ===")
con.sql("SELECT * FROM dim_clients LIMIT 5").show()

print("\n=== DIM_CONTENT ===")
con.sql("SELECT * FROM dim_content LIMIT 5").show()

print("\n=== FACT_DAILY ===")
con.sql("SELECT * FROM fact_daily LIMIT 5").show()

print("\n=== FACT_QUERY_90D ===")
con.sql("SELECT * FROM fact_query_90d LIMIT 5").show()

=== DIM_CLIENTS ===
┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ fal

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [23]:
print("FACT_DAILY date range:")
con.sql("""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM fact_daily
""").show()

print("QUERY 90D date range:")
con.sql("""
    SELECT
        MIN(window_start) AS min_date,
        MAX(window_end) AS max_date
    FROM fact_query_90d
""").show()

FACT_DAILY date range:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│  min_date  │  max_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-01-27 │ 2026-06-30 │
└────────────┴────────────┘

QUERY 90D date range:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│  min_date  │  max_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-04-02 │ 2026-06-30 │
└────────────┴────────────┘



In [25]:
con.sql("""
SELECT
    SUM(gsc_impressions) AS total_gsc_impressions,
    SUM(gsc_clicks) AS total_gsc_clicks,
    SUM(sessions_ai) AS total_ai_sessions,
    SUM(sessions_paid) AS total_paid_sessions,
    SUM(sessions_direct) AS total_direct_sessions,
    SUM(sessions_social) AS total_social_sessions,
    SUM(sessions_organic) AS total_organic_sessions
FROM fact_daily
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────┬──────────────────┬───────────────────┬─────────────────────┬───────────────────────┬───────────────────────┬────────────────────────┐
│ total_gsc_impressions │ total_gsc_clicks │ total_ai_sessions │ total_paid_sessions │ total_direct_sessions │ total_social_sessions │ total_organic_sessions │
│        int128         │      int128      │      int128       │       int128        │        int128         │        int128         │         int128         │
├───────────────────────┼──────────────────┼───────────────────┼─────────────────────┼───────────────────────┼───────────────────────┼────────────────────────┤
│            1763519980 │          6440837 │             83618 │             1121640 │               2035417 │                149286 │                5213396 │
└───────────────────────┴──────────────────┴───────────────────┴─────────────────────┴───────────────────────┴───────────────────────┴────────────────────────┘



In [26]:
con.sql("""
SELECT
    SUM(sessions_ai) AS ai_sessions,
    SUM(ai_chatgpt) AS chatgpt_sessions,
    SUM(ai_perplexity) AS perplexity_sessions,
    SUM(ai_gemini) AS gemini_sessions,
    SUM(ai_copilot) AS copilot_sessions,
    SUM(ai_claude) AS claude_sessions,
    SUM(ai_meta) AS meta_sessions,
    SUM(ai_other) AS other_ai_sessions
FROM fact_daily
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬──────────────────┬─────────────────────┬─────────────────┬──────────────────┬─────────────────┬───────────────┬───────────────────┐
│ ai_sessions │ chatgpt_sessions │ perplexity_sessions │ gemini_sessions │ copilot_sessions │ claude_sessions │ meta_sessions │ other_ai_sessions │
│   int128    │      int128      │       int128        │     int128      │      int128      │     int128      │    int128     │      int128       │
├─────────────┼──────────────────┼─────────────────────┼─────────────────┼──────────────────┼─────────────────┼───────────────┼───────────────────┤
│       83618 │            60299 │                7061 │           13425 │             1983 │             888 │             1 │                 0 │
└─────────────┴──────────────────┴─────────────────────┴─────────────────┴──────────────────┴─────────────────┴───────────────┴───────────────────┘



In [13]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Token loaded:", bool(HF_TOKEN))
print("Token starts with hf_:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)

Token loaded: True
Token starts with hf_: True


In [11]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    'Paste your Hugging Face READ token (hf_...): '
)

print("HF_TOKEN exists:", bool(HF_TOKEN))
print("HF_TOKEN starts with hf_:", HF_TOKEN.startswith("hf_"))

HF_TOKEN exists: True
HF_TOKEN starts with hf_: True


In [15]:
print("Token loaded:", bool(HF_TOKEN))
print("Token starts with hf_:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)

Token loaded: True
Token starts with hf_: True


In [28]:
from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Download successful:", test_file)

Download successful: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [12]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [6]:
con.sql("""
    SELECT COUNT(*)
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
    )
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          104 │
└──────────────┘



That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [7]:
for name, src in TABLES.items():
    print(f"\n=== {name} ===")
    con.sql(f"DESCRIBE SELECT * FROM {src}").show()


=== dim_clients ===
┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES  

In [8]:
for name, src in TABLES.items():
    print(f"\n=== {name} ===")
    con.sql(f"SELECT * FROM {src} LIMIT 3").show()


=== dim_clients ===
┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ fa

In [33]:
con.sql("""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS total_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ first_date │ last_date  │ total_rows │
│    date    │    date    │   int64    │
├────────────┼────────────┼────────────┤
│ 2025-01-27 │ 2026-06-30 │   78835655 │
└────────────┴────────────┴────────────┘



In [36]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [13]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [15]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,

        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"]
    / qsignals["kept_impressions"]
)

data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print(f"joined: {len(data):,} rows")

data.head()

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [16]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.549     0.339     0.420      9389
           1      0.686     0.838     0.754     16162

    accuracy                          0.655     25551
   macro avg      0.617     0.589     0.587     25551
weighted avg      0.636     0.655     0.631     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [17]:
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev60,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_avg_position
                END
            ) AS avg_position_90d

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev60 >= 300
    )

    SELECT *
    FROM windowed
""").df()

print(f"{len(features_90d):,} content items with enough history")
features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,044 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev60,avg_position_90d
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,1060.0,8.642429
1,client_e547b89c05043229,content_bab118937886d46a,110.0,359.0,21.897400
2,client_e547b89c05043229,content_a64be0ba11772089,248.0,857.0,15.264068
3,client_e547b89c05043229,content_74369d7d3369d86b,192.0,1136.0,17.584280
4,client_e547b89c05043229,content_1e1be93551c6cb91,629.0,1398.0,16.598413


In [18]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data.dropna(subset=feature_cols + ["client_hash_id"]).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]
y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]

print("Training rows:", len(X_tr))
print("Test rows:", len(X_te))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 50501
Test rows: 51702
Training clients: 36
Test clients: 13


In [19]:
model_group = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_group.fit(X_tr, y_tr)

pred_group = model_group.predict(X_te)

print(
    f"Group-split base rate: "
    f"{max(y_te.mean(), 1 - y_te.mean()):.3f}"
)

print(
    classification_report(
        y_te,
        pred_group,
        digits=3
    )
)

Group-split base rate: 0.677
              precision    recall  f1-score   support

           0      0.412     0.462     0.436     16706
           1      0.727     0.685     0.706     34996

    accuracy                          0.613     51702
   macro avg      0.570     0.574     0.571     51702
weighted avg      0.626     0.613     0.618     51702



In [20]:
import pandas as pd

In [21]:
importance = (
    pd.Series(
        model_group.feature_importances_,
        index=feature_cols
    )
    .sort_values(ascending=False)
)

print(importance)

rare_share         0.235455
anon_share         0.234547
imp_prev30         0.222099
top_query_share    0.191766
visible_queries    0.116133
dtype: float64


In [22]:
position_volatility = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    )

    SELECT
        client_hash_id,
        content_hash_id,
        STDDEV_SAMP(gsc_avg_position) AS position_volatility

    FROM {TABLES['fact_daily']} f, bounds b

    WHERE f.report_date > b.end_d - INTERVAL 30 DAY

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print(f"{len(position_volatility):,} content items with position volatility")

position_volatility.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

409,205 content items with position volatility


,client_hash_id,content_hash_id,position_volatility
0,client_e547b89c05043229,content_f1d30b8ab90f83ec,2.663712
1,client_e547b89c05043229,content_bfbea1a8bd407b8a,25.488342
2,client_e547b89c05043229,content_3f51d78fdd2de339,18.826172
3,client_e547b89c05043229,content_3f312fa32a603af9,15.611918
4,client_e547b89c05043229,content_cff9bdab98d62b0a,10.639549


In [23]:
data = data.merge(
    position_volatility,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(data.shape)

(111247, 14)


In [24]:
feature_cols_extra = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "position_volatility"
]

model_data_extra = data.dropna(
    subset=feature_cols_extra + ["client_hash_id"]
).copy()

X = model_data_extra[feature_cols_extra]
y = model_data_extra["is_declining"]
groups = model_data_extra["client_hash_id"]

In [25]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]
y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]

model_extra = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_extra.fit(X_tr, y_tr)

pred_extra = model_extra.predict(X_te)

print(classification_report(
    y_te,
    pred_extra,
    digits=3
))

              precision    recall  f1-score   support

           0      0.625     0.645     0.635     10849
           1      0.793     0.779     0.786     18932

    accuracy                          0.730     29781
   macro avg      0.709     0.712     0.710     29781
weighted avg      0.732     0.730     0.731     29781



In [26]:
importance_extra = (
    pd.Series(
        model_extra.feature_importances_,
        index=feature_cols_extra
    )
    .sort_values(ascending=False)
)

print(importance_extra)

position_volatility    0.309295
imp_prev30             0.181043
rare_share             0.160552
anon_share             0.146510
top_query_share        0.118427
visible_queries        0.084173
dtype: float64


In [27]:
print("=== RANDOM SPLIT ===")
print(classification_report(
    y_te if False else y_te,
    pred_extra,
    digits=3
))

=== RANDOM SPLIT ===
              precision    recall  f1-score   support

           0      0.625     0.645     0.635     10849
           1      0.793     0.779     0.786     18932

    accuracy                          0.730     29781
   macro avg      0.709     0.712     0.710     29781
weighted avg      0.732     0.730     0.731     29781



In [29]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

feature_cols_extra = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "position_volatility"
]

model_data_extra = data.dropna(
    subset=feature_cols_extra + ["client_hash_id"]
).copy()

X = model_data_extra[feature_cols_extra]
y = model_data_extra["is_declining"]
groups = model_data_extra["client_hash_id"]

# -----------------------------
# Random split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_train, y_train)
random_pred = random_model.predict(X_test)

# -----------------------------
# Client-level split
# -----------------------------
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

group_pred = group_model.predict(
    X.iloc[test_idx]
)

# -----------------------------
# Comparison
# -----------------------------
print("RANDOM SPLIT")
print("Accuracy :", round(accuracy_score(y_test, random_pred), 3))
print("Precision:", round(precision_score(y_test, random_pred), 3))
print("Recall   :", round(recall_score(y_test, random_pred), 3))
print("F1       :", round(f1_score(y_test, random_pred), 3))

print("\nCLIENT-LEVEL SPLIT")
print("Accuracy :", round(
    accuracy_score(y.iloc[test_idx], group_pred), 3
))
print("Precision:", round(
    precision_score(y.iloc[test_idx], group_pred), 3
))
print("Recall   :", round(
    recall_score(y.iloc[test_idx], group_pred), 3
))
print("F1       :", round(
    f1_score(y.iloc[test_idx], group_pred), 3
))

RANDOM SPLIT
Accuracy : 0.754
Precision: 0.787
Recall   : 0.835
F1       : 0.81

CLIENT-LEVEL SPLIT
Accuracy : 0.73
Precision: 0.793
Recall   : 0.779
F1       : 0.786


In [30]:
importance_final = (
    pd.Series(
        group_model.feature_importances_,
        index=feature_cols_extra
    )
    .sort_values(ascending=False)
)

print("FEATURE IMPORTANCE")
print(importance_final)

FEATURE IMPORTANCE
position_volatility    0.309295
imp_prev30             0.181043
rare_share             0.160552
anon_share             0.146510
top_query_share        0.118427
visible_queries        0.084173
dtype: float64


In [31]:
print("=== FINAL DATASET ===")
print(f"Rows used: {len(model_data_extra):,}")
print(f"Features: {feature_cols_extra}")

print("\n=== RANDOM SPLIT ===")
print(f"Accuracy : {accuracy_score(y_test, random_pred):.3f}")
print(f"Precision: {precision_score(y_test, random_pred):.3f}")
print(f"Recall   : {recall_score(y_test, random_pred):.3f}")
print(f"F1       : {f1_score(y_test, random_pred):.3f}")

print("\n=== CLIENT-LEVEL SPLIT ===")
print(f"Accuracy : {accuracy_score(y.iloc[test_idx], group_pred):.3f}")
print(f"Precision: {precision_score(y.iloc[test_idx], group_pred):.3f}")
print(f"Recall   : {recall_score(y.iloc[test_idx], group_pred):.3f}")
print(f"F1       : {f1_score(y.iloc[test_idx], group_pred):.3f}")

print("\n=== FEATURE IMPORTANCE ===")
print(importance_final)

=== FINAL DATASET ===
Rows used: 101,251
Features: ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility']

=== RANDOM SPLIT ===
Accuracy : 0.754
Precision: 0.787
Recall   : 0.835
F1       : 0.810

=== CLIENT-LEVEL SPLIT ===
Accuracy : 0.730
Precision: 0.793
Recall   : 0.779
F1       : 0.786

=== FEATURE IMPORTANCE ===
position_volatility    0.309295
imp_prev30             0.181043
rare_share             0.160552
anon_share             0.146510
top_query_share        0.118427
visible_queries        0.084173
dtype: float64


In [32]:
required_objects = [
    "features",
    "qsignals",
    "data",
    "features_90d",
    "model_data",
    "model_data_extra",
    "random_model",
    "group_model",
    "importance_final"
]

for name in required_objects:
    print(f"{name:20} ->", "OK" if name in globals() else "MISSING")

features             -> OK
qsignals             -> OK
data                 -> OK
features_90d         -> OK
model_data           -> OK
model_data_extra     -> OK
random_model         -> OK
group_model          -> OK
importance_final     -> OK


In [33]:
print("Target distribution:")
print(model_data_extra["is_declining"].value_counts())

print("\nTarget proportions:")
print(model_data_extra["is_declining"].value_counts(normalize=True))

Target distribution:
is_declining
1    63694
0    37557
Name: count, dtype: int64

Target proportions:
is_declining
1    0.62907
0    0.37093
Name: proportion, dtype: float64


In [34]:
final_importance = pd.DataFrame({
    "feature": importance_final.index,
    "importance": importance_final.values
})

final_importance

,feature,importance
0,position_volatility,0.309295
1,imp_prev30,0.181043
2,rare_share,0.160552
3,anon_share,0.146510
4,top_query_share,0.118427
5,visible_queries,0.084173
